# Week 03 · Attention in Action
## Train a Tiny Storyteller & Meet our Own SLM

Pre-trained   → learned language from billions of words before you touched it

Fine-tuned    → you nudge it toward your specific task or style

Attention     → the mechanism it uses to decide which words matter in context

Context window→ the working memory it reads and writes within

Parameters    → the numbers that store everything it learned — GPT-4 has ~1 trillion

**What you will understand by the end of this session:**

| Concept | Why it matters for your work |
|---------|-----------------------------|
| The context window | Everything in RAG, Agents & MCP goes through this |
| Self-attention | How the model reads and connects words |
| Causal mask | Why generation is word-by-word and left-to-right |
| Multi-head attention | How one model notices grammar, sentiment & facts simultaneously |
| Cross-attention | How agents read tool results — Week 6 preview |
| Generation strategies | Temperature, top-k, greedy — the dials you already use |

**No deep ML knowledge needed. Every output is plain text. Every concept has one sentence explanation.**

---
> **The session question:**  
> *"I asked the AI to recommend a restaurant near the Eiffel Tower and it made one up. Why? And how do we fix it?"*  
> We answer this fully by the end.

## 0 · Setup

In [2]:
# Install — pin huggingface_hub to avoid XetHub upload bug
# Must be the very first cell before any other imports
import sys
!{sys.executable} -m pip install -q "huggingface_hub==1.12.2"
!{sys.executable} -m pip install -q transformers torch matplotlib IPython


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\vivik\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\vivik\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
import math, warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')   
torch.manual_seed(42)
plt.rcParams.update({'font.size': 11, 'figure.dpi': 110})

print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {"GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print()
print('✅ Ready')

PyTorch  : 2.12.1+cpu
Device   : CPU

✅ Ready


---
## Act 1 · The Context Window

**The single most important mental model in this programme.**

An LLM has no persistent memory. No database. No internet access by default.  
All it has is a **context window** — a box of tokens it can currently see.

> Everything you will learn in Weeks 4–8 is about what you put in that box.  
> RAG puts retrieved documents in the box.  
> Agents put tool results in the box.  
> MCP standardises how things get into the box.

In [6]:
"""
Demonstrate the context window visually.

The model can only use what is inside the window.
Everything outside does not exist from its perspective.
This is why it hallucinates — it fills gaps with probability, not facts.
"""

def show_context_window(context_items, window_limit=10, title="Context Window"):
    print(f"\n{'═'*58}")
    print(f"  {title}  (limit: {window_limit} items shown)")
    print(f"{'═'*58}")

    for i, item in enumerate(context_items[:window_limit]):
        tag, text = item
        status = '✓ INSIDE' if i < window_limit else '✗ OUTSIDE'
        print(f"  [{tag:<12}]  {text[:50]}")

    if len(context_items) > window_limit:
        overflow = len(context_items) - window_limit
        print(f"  {'─'*54}")
        print(f"  ✗ {overflow} items OUTSIDE window — model cannot see these")

    print(f"{'═'*58}")

# Scenario 1: Bare question with no grounding
bare_context = [
    ("SYSTEM",    "You are a helpful assistant."),
    ("USER",      "Recommend a restaurant near the Eiffel Tower."),
]

print("SCENARIO 1 — No grounding data in the context window")
show_context_window(bare_context, title="Context Window (bare)")
print()
print("  Model response (likely):")
display(Markdown("Le Jules Verne is a wonderful choice, located on the second floor of the Eiffel Tower with stunning views..."))
print("─" * 58)
display(Markdown("Problem: Le Jules Verne exists or does not possible but the model may confabulate, pening hours, prices, availability. It fills gaps with probability, not facts. This is hallucination."))
print("─" * 58)
print()

# Scenario 2: RAG fills the window with real data
rag_context = [
    ("SYSTEM",    "You are a helpful assistant. Use only the provided data."),
    ("RETRIEVED", "Le Jules Verne: Eiffel Tower 2nd floor. Open Tue-Sun."),
    ("RETRIEVED", "Café de Flore: 172 Blvd Saint-Germain. Avg €45/person."),
    ("RETRIEVED", "Brasserie Thoumieux: 79 Rue Saint-Dominique. Open daily."),
    ("USER",      "Recommend a restaurant near the Eiffel Tower."),
]

print("SCENARIO 2 — RAG fills the window with real retrieved data")
show_context_window(rag_context, title="Context Window (with RAG)")
print("─" * 58)
print(" Model response (grounded):")
display(Markdown(" 'Based on the provided files, Le Jules Verne is located on the Eiffel Tower's 2nd floor and is open Tuesday–Sunday"))
print("─" * 58)
print(" Same model, same question. Different context = different answer.")
print(" Week 4 is entirely about building the retrieval that fills this window.")

SCENARIO 1 — No grounding data in the context window

══════════════════════════════════════════════════════════
  Context Window (bare)  (limit: 10 items shown)
══════════════════════════════════════════════════════════
  [SYSTEM      ]  You are a helpful assistant.
  [USER        ]  Recommend a restaurant near the Eiffel Tower.
══════════════════════════════════════════════════════════

  Model response (likely):


Le Jules Verne is a wonderful choice, located on the second floor of the Eiffel Tower with stunning views...

──────────────────────────────────────────────────────────


Problem: Le Jules Verne exists or does not possible but the model may confabulate, pening hours, prices, availability. It fills gaps with probability, not facts. This is hallucination.

──────────────────────────────────────────────────────────

SCENARIO 2 — RAG fills the window with real retrieved data

══════════════════════════════════════════════════════════
  Context Window (with RAG)  (limit: 10 items shown)
══════════════════════════════════════════════════════════
  [SYSTEM      ]  You are a helpful assistant. Use only the provided
  [RETRIEVED   ]  Le Jules Verne: Eiffel Tower 2nd floor. Open Tue-S
  [RETRIEVED   ]  Café de Flore: 172 Blvd Saint-Germain. Avg €45/per
  [RETRIEVED   ]  Brasserie Thoumieux: 79 Rue Saint-Dominique. Open 
  [USER        ]  Recommend a restaurant near the Eiffel Tower.
══════════════════════════════════════════════════════════
──────────────────────────────────────────────────────────
 Model response (grounded):


 'Based on the provided files, Le Jules Verne is located on the Eiffel Tower's 2nd floor and is open Tuesday–Sunday

──────────────────────────────────────────────────────────
 Same model, same question. Different context = different answer.
 Week 4 is entirely about building the retrieval that fills this window.


---
## Act 2 · Attention — Five Types, Plain Text, Real Weights

We use a small trained model to extract **real attention weights** 

The output format is always the same: word on the left, weight as a number, bar shows relative focus.  
**Longer bar = model pays more attention to that word.**


A variant of attention mechanism which reduces reliance on external info and excels at capturing internal correlations within data or features. It has 3 steps:
1. query vector — word being attended to
2. key vectors — all the words in sentence.
3. value vectors — store info associated with each word.

Attention weights are computed by taking dot product between query and key vectors, followed by softmax operation to obtain distribution over words.

In [ ]:
"""
Small Training for Vocab
"""

VOCAB = [
    '<PAD>', '<UNK>', '<BOS>', '<EOS>',
    # fairy tale words
    'once', 'upon', 'a', 'time', 'there', 'was', 'the', 'little',
    'prince', 'princess', 'knight', 'dragon', 'castle', 'forest',
    'dark', 'old', 'brave', 'kind', 'beautiful', 'wicked',
    'rode', 'found', 'looked', 'said', 'lived', 'searched',
    'and', 'but', 'in', 'through', 'by', 'near', 'on', 'with',
    # sentiment words
    'fantastic', 'brilliant', 'terrible', 'boring', 'magical', 'dull',
    # story words
    'bank', 'river', 'water', 'ice', 'snow', 'gold', 'spell', 'heart',
    'happily', 'ever', 'after', 'her', 'his', 'him', 'she', 'he',
]
v2i = {w: i for i, w in enumerate(VOCAB)}
i2v = {i: w for i, w in enumerate(VOCAB)}
V   = len(VOCAB)

def encode(words):
    return [v2i.get(w, v2i['<UNK>']) for w in words]


#  Model definition
class MultiHeadAttention(nn.Module):
    """
    Multi-head self-attention.
    Runs H parallel attention computations, each looking at different aspects.
    Stores weights after each forward pass for inspection.
    """
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.h  = n_heads
        self.dk = d_model // n_heads
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)
        self.last_weights = None   

    def forward(self, x, mask=None):
        B, S, D = x.shape
        def split(t): return t.view(B, S, self.h, self.dk).transpose(1, 2)
        Q, K, V_ = split(self.Wq(x)), split(self.Wk(x)), split(self.Wv(x))
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.dk)
        if mask is not None:
            scores = scores.masked_fill(mask[None, None], float('-inf'))
        w = F.softmax(scores, dim=-1)
        self.last_weights = w.detach()   # shape: (B, H, S, S)
        out = (w @ V_).transpose(1, 2).contiguous().view(B, S, D)
        return self.Wo(out)


class EncoderBlock(nn.Module):
    def __init__(self, d, h, ff=None):
        super().__init__()
        self.attn = MultiHeadAttention(d, h)
        self.n1   = nn.LayerNorm(d)
        self.n2   = nn.LayerNorm(d)
        self.ff   = nn.Sequential(
            nn.Linear(d, ff or d*2), nn.GELU(), nn.Linear(ff or d*2, d)
        )

    def forward(self, x, mask=None):
        x = self.n1(x + self.attn(x, mask=mask))
        return self.n2(x + self.ff(x))


class TinyLM(nn.Module):
    """
    Small GPT-style language model.
    Causal mask is applied in every block — each position only sees past tokens.
    """
    def __init__(self, vocab_size, d=64, h=4, layers=2, max_len=32):
        super().__init__()
        self.te     = nn.Embedding(vocab_size, d, padding_idx=0)
        self.pe     = nn.Embedding(max_len, d)
        self.blocks = nn.ModuleList([EncoderBlock(d, h) for _ in range(layers)])
        self.norm   = nn.LayerNorm(d)
        self.head   = nn.Linear(d, vocab_size, bias=False)
        self.d      = d

    def forward(self, idx, use_causal_mask=True):
        B, S = idx.shape
        mask = None
        if use_causal_mask:
            mask = torch.triu(torch.ones(S, S, dtype=torch.bool), diagonal=1)
        pos  = torch.arange(S).unsqueeze(0)
        x    = self.te(idx) * math.sqrt(self.d) + self.pe(pos)
        for block in self.blocks:
            x = block(x, mask=mask)
        return self.head(self.norm(x))

    def get_attention(self, layer=-1):
        """Return attention weights from a specific layer."""
        return self.blocks[layer].attn.last_weights


# Quick training on fairy tale sentences
SENTENCES = [
    "once upon a time the brave knight rode through the dark forest",
    "the little prince searched the old castle near the river bank",
    "she found him and her heart felt magical and beautiful",
    "the wicked dragon guarded the gold with a terrible spell",
    "they lived happily ever after in the castle by the water",
    "the brave princess rode through the forest with her kind heart",
    "once there was a brilliant knight who found the castle dark",
    "the fantastic spell broke and she lived happily ever after",
]

def make_batch(sentences, max_len=20):
    xs, ys = [], []
    for s in sentences:
        words = ['<BOS>'] + s.split() + ['<EOS>']
        ids   = encode(words)[:max_len]
        ids  += [0] * (max_len - len(ids))
        xs.append(ids[:-1])
        ys.append(ids[1:])
    return torch.tensor(xs), torch.tensor(ys)

tiny = TinyLM(V, d=64, h=4, layers=2, max_len=20)
X, Y = make_batch(SENTENCES)

opt  = torch.optim.AdamW(tiny.parameters(), lr=5e-3)
crit = nn.CrossEntropyLoss(ignore_index=0)

print("Quick training on fairy tale sentences (30 epochs)...")
for ep in range(30):
    tiny.train()
    logits = tiny(X)
    loss   = crit(logits.view(-1, V), Y.view(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if (ep + 1) % 10 == 0:
        print(f"  Epoch {ep+1:3d}  loss = {loss.item():.4f}")

tiny.eval()
total = sum(p.numel() for p in tiny.parameters())
print(f"\n✅ TinyLM ready  |  {total:,} parameters  |  4 attention heads  |  2 layers")

Quick training on fairy tale sentences (30 epochs)...
  Epoch  10  loss = 0.9021
  Epoch  20  loss = 0.2583
  Epoch  30  loss = 0.1988

✅ TinyLM ready  |  75,520 parameters  |  4 attention heads  |  2 layers


In [44]:
def inspect_attention(
    sentence_words,
    query_word,
    title,
    layer    = -1,
    head     = 0,
    use_mask = True,
    blocked  = None,
):
    """
    Run a sentence through TinyLM, extract real attention weights,
    and print them as plain text with block-character bars.

    FIX: <BOS> token is excluded from display — it acts as an
    attention sink and dominates weights in untrained/small models.
    Remaining weights are renormalised to sum to 1.
    """
    ids        = torch.tensor([encode(['<BOS>'] + sentence_words)])
    full_words = ['<BOS>'] + sentence_words

    with torch.no_grad():
        _ = tiny(ids, use_causal_mask=use_mask)

    weights = tiny.get_attention(layer)   # (1, H, S, S)
    S       = len(full_words)

    q_idx = next(
        (i for i, w in enumerate(full_words) if w == query_word), -1
    )
    if q_idx == -1:
        print(f"  '{query_word}' not found. Available: {sentence_words}")
        return

    row = weights[0, head, q_idx, :S].numpy().copy()

    # Zero out <BOS> — attention sink, not a meaningful word
    row[0] = 0.0

    # Zero out any manually blocked positions (padding demo)
    if blocked:
        for idx in blocked:
            row[idx] = 0.0

    # Renormalise so remaining weights sum to 1
    total = row.sum()
    if total > 1e-6:
        row = row / total

    # Skip <BOS> in display entirely
    display_words = sentence_words
    display_row   = row[1:len(sentence_words) + 1]
    col_w         = max(len(w) for w in display_words)
    top_idx       = display_row.argmax()

    print(f"\n{'═'*58}")
    print(f"  {title}")
    print(f"{'═'*58}")
    print(f"  Sentence   : \"{' '.join(sentence_words)}\"")
    print(f"  Query word : \"{query_word}\"  (head {head+1})")
    print()
    print(f"  {'Word':<{col_w}}   Weight  Bar")
    print(f"  {'─'*50}")
    for j, (word, weight) in enumerate(zip(display_words, display_row)):
        bar    = '█' * int(weight * 25)
        marker = '  ← focus' if j == top_idx and weight > 0.01 else ''
        block  = '  [BLOCKED]' if blocked and (j+1) in (blocked or []) else ''
        print(f"  {word:<{col_w}}   {weight:.3f}   {bar}{marker}{block}")
    print(f"  {'─'*50}")
    print(f"  Model focuses most on: \"{display_words[top_idx]}\"")
    print()

### Type 1 · Self-Attention
**"Which words help me understand this word?"**

Every word attends to every other word. No restrictions.  
→ *Week 4 RAG: when a retrieved document is read, self-attention connects its concepts.*

In [43]:
"""
Self-attention: every word can see every other word.
Here we ask: when the model reads 'bank', which words help it understand what kind of bank?
Two identical words in two different contexts — attention should differ.
"""

# Context 1: river bank
inspect_attention(
    sentence_words = ['the', 'prince', 'rode', 'near', 'the', 'river', 'bank'],
    query_word     = 'river',
    title          = "TYPE 1 · SELF-ATTENTION  |  'river' in geographic context",
    use_mask       = False,   # self-attention — no causal restriction
    head           = 0,
)

# Context 2: castle context  
inspect_attention(
    sentence_words = ['she', 'found', 'the', 'dark', 'old', 'castle', 'near', 'the', 'forest'],
    query_word     = 'castle',
    title          = "TYPE 1 · SELF-ATTENTION  |  'castle' — what does the model focus on?",
    use_mask       = False,
    head           = 0,
)

print("💡 Self-attention has no restrictions — every word sees every other word.")
print("   This is used in the ENCODER side of models (BERT, RAG embeddings).")
print("   When your RAG system converts a document to a vector,")
print("   self-attention is how word relationships are captured.")


══════════════════════════════════════════════════════════
  TYPE 1 · SELF-ATTENTION  |  'river' in geographic context
══════════════════════════════════════════════════════════
  Sentence   : "the prince rode near the river bank"
  Query word : "river"  (head 1)

  Word     Weight  Bar
  ──────────────────────────────────────────────────
  the      0.014   
  prince   0.063   █
  rode     0.302   ███████  ← focus
  near     0.207   █████
  the      0.014   
  river    0.197   ████
  bank     0.203   █████
  ──────────────────────────────────────────────────
  Model focuses most on: "rode"


══════════════════════════════════════════════════════════
  TYPE 1 · SELF-ATTENTION  |  'castle' — what does the model focus on?
══════════════════════════════════════════════════════════
  Sentence   : "she found the dark old castle near the forest"
  Query word : "castle"  (head 1)

  Word     Weight  Bar
  ──────────────────────────────────────────────────
  she      0.531   █████████████  ← f

### Type 2 · Causal (Masked) Attention
**"How do I predict the next word without cheating?"**

Each word can only see words before it. The future is blocked.  
→ *Every generative model you use — GPT, Claude, Gemini — runs this at every step.*

In [45]:
"""
Causal mask: position i can only attend to positions 0..i.
Show the mask explicitly as a grid, then show it in action on a sentence.
This is the mechanism behind every word ChatGPT has ever generated.
"""

def show_causal_mask_grid(words):
    """Print the causal mask as a ✓/✗ grid — who can see whom."""
    n     = len(words)
    col_w = max(len(w) for w in words)

    print(f"\n{'═'*58}")
    print(f"  CAUSAL MASK — What each word is ALLOWED to see")
    print(f"{'═'*58}")
    print(f"  Sentence: \"{' '.join(words)}\"")
    print()

    # Header row
    print(f"  {'Word':<{col_w+2}}", end="")
    for w in words: print(f"  {w[:5]:>5}", end="")
    print()
    print(f"  {'─'*(col_w + 2 + n*7)}")

    for i, qw in enumerate(words):
        print(f"  {qw:<{col_w+2}}", end="")
        for j in range(n):
            sym = ' ✓ ' if j <= i else ' ✗ '
            print(f"  {sym:>5}", end="")
        print()

    print()
    print("  ✓ = allowed (past + current)   ✗ = BLOCKED (future)")
    print(f"{'═'*58}")


words_gen = ["once", "upon", "a", "time", "there", "was"]
show_causal_mask_grid(words_gen)

print()
print("Now see it in action — with vs without the causal mask:")

# With causal mask
inspect_attention(
    sentence_words = words_gen,
    query_word     = 'time',
    title          = "TYPE 2 · CAUSAL MASK ON  |  'time' cannot see 'there' or 'was'",
    use_mask       = True,
    head           = 0,
)

# Without causal mask (for comparison)
inspect_attention(
    sentence_words = words_gen,
    query_word     = 'time',
    title          = "NO MASK (comparison)  |  'time' cheats — sees 'there' and 'was'",
    use_mask       = False,
    head           = 0,
)

print("💡 The causal mask is why generation is word-by-word.")
print("   Without it the model would see the answer during training.")
print("   It would learn nothing — like doing an exam with the answer sheet.")
print()
print("   GPT, Claude, Gemini — all use this mask for every word they generate.")


══════════════════════════════════════════════════════════
  CAUSAL MASK — What each word is ALLOWED to see
══════════════════════════════════════════════════════════
  Sentence: "once upon a time there was"

  Word      once   upon      a   time  there    was
  ─────────────────────────────────────────────────
  once        ✓      ✗      ✗      ✗      ✗      ✗ 
  upon        ✓      ✓      ✗      ✗      ✗      ✗ 
  a           ✓      ✓      ✓      ✗      ✗      ✗ 
  time        ✓      ✓      ✓      ✓      ✗      ✗ 
  there       ✓      ✓      ✓      ✓      ✓      ✗ 
  was         ✓      ✓      ✓      ✓      ✓      ✓ 

  ✓ = allowed (past + current)   ✗ = BLOCKED (future)
══════════════════════════════════════════════════════════

Now see it in action — with vs without the causal mask:

══════════════════════════════════════════════════════════
  TYPE 2 · CAUSAL MASK ON  |  'time' cannot see 'there' or 'was'
══════════════════════════════════════════════════════════
  Sentence   : "onc

### Type 3 · Multi-Head Attention
**"Can one model notice grammar, sentiment and facts simultaneously?"**

Run attention H times in parallel. Each head learns to focus on different patterns.  
→ *This is why a single LLM can track facts, tone and format in one pass.*

In [46]:
"""
Multi-head attention: same sentence, same word, four different heads.
Each head has learned to focus on different linguistic relationships.
Together they give a richer representation than any single head could.
"""

sentence = ['the', 'brave', 'knight', 'rode', 'through', 'the', 'dark', 'forest']
query    = 'knight'

print(f'Sentence: "{" ".join(sentence)}"')
print(f'Query word: "{query}" — four heads, four perspectives\n')

head_labels = [
    "Head 1 — what kind of knight?",
    "Head 2 — what is the knight doing?",
    "Head 3 — where is the knight going?",
    "Head 4 — what is the overall tone?",
]

for h, label in enumerate(head_labels):
    inspect_attention(
        sentence_words = sentence,
        query_word     = query,
        title          = f"TYPE 3 · MULTI-HEAD  |  {label}",
        use_mask       = False,
        head           = h,
    )

print("💡 Each head sees the same sentence but focuses on different patterns.")
print("   This is why one model can simultaneously track:")
print("     - the facts in your RAG document")
print("     - the tone of your question")
print("     - the format you asked for")
print("   All in a single forward pass.")

Sentence: "the brave knight rode through the dark forest"
Query word: "knight" — four heads, four perspectives


══════════════════════════════════════════════════════════
  TYPE 3 · MULTI-HEAD  |  Head 1 — what kind of knight?
══════════════════════════════════════════════════════════
  Sentence   : "the brave knight rode through the dark forest"
  Query word : "knight"  (head 1)

  Word      Weight  Bar
  ──────────────────────────────────────────────────
  the       0.187   ████
  brave     0.004   
  knight    0.013   
  rode      0.077   █
  through   0.056   █
  the       0.197   ████
  dark      0.019   
  forest    0.447   ███████████  ← focus
  ──────────────────────────────────────────────────
  Model focuses most on: "forest"


══════════════════════════════════════════════════════════
  TYPE 3 · MULTI-HEAD  |  Head 2 — what is the knight doing?
══════════════════════════════════════════════════════════
  Sentence   : "the brave knight rode through the dark forest"
  Query w

### Type 4 · Padding Mask
**"How does the model handle sentences of different lengths in a batch?"**

Short sentences are padded to match the longest. The model must ignore `<PAD>` tokens.  
→ *Every time you send a batch of prompts to an API, padding masks are running.*

In [47]:
"""
Padding mask: sentences in a batch must be the same length.
Shorter sentences are padded with <PAD> tokens.
The model must be told to ignore these — they contain no real information.
"""

sentence_with_pads = ['the', 'brave', 'knight', '<PAD>', '<PAD>', '<PAD>']
pad_positions      = [4, 5, 6]   # positions of <PAD> in full sequence incl <BOS>

print("Sentence: 'the brave knight' padded to length 6")
print("Padding positions (indices in full sequence): ", pad_positions)
print()

inspect_attention(
    sentence_words = sentence_with_pads,
    query_word     = 'knight',
    title          = "TYPE 4 · NO PADDING MASK  |  attention leaks into <PAD>",
    use_mask       = False,
)

inspect_attention(
    sentence_words = sentence_with_pads,
    query_word     = 'knight',
    title          = "TYPE 4 · WITH PADDING MASK  |  <PAD> tokens zeroed out",
    use_mask       = False,
    blocked        = pad_positions,
)

print("💡 Without the mask: some attention is wasted on empty padding.")
print("   With the mask: all attention redistributes to real words.")
print("   This happens automatically in every API call you make.")

Sentence: 'the brave knight' padded to length 6
Padding positions (indices in full sequence):  [4, 5, 6]


══════════════════════════════════════════════════════════
  TYPE 4 · NO PADDING MASK  |  attention leaks into <PAD>
══════════════════════════════════════════════════════════
  Sentence   : "the brave knight <PAD> <PAD> <PAD>"
  Query word : "knight"  (head 1)

  Word     Weight  Bar
  ──────────────────────────────────────────────────
  the      0.138   ███
  brave    0.067   █
  knight   0.196   ████
  <PAD>    0.125   ███
  <PAD>    0.181   ████
  <PAD>    0.294   ███████  ← focus
  ──────────────────────────────────────────────────
  Model focuses most on: "<PAD>"


══════════════════════════════════════════════════════════
  TYPE 4 · WITH PADDING MASK  |  <PAD> tokens zeroed out
══════════════════════════════════════════════════════════
  Sentence   : "the brave knight <PAD> <PAD> <PAD>"
  Query word : "knight"  (head 1)

  Word     Weight  Bar
  ────────────────────────────

### Type 5 · Cross-Attention
**"How does the model read a tool result while writing its reply?"**

Query comes from one sequence (what the model is generating).  
Keys and values come from another (the tool result or retrieved document).  
→ *This is the conceptual engine of agents and RAG — Weeks 4, 6, 7, 8.*

In [48]:
"""
Cross-attention: the query comes from the generation stream,
the keys/values come from an external source (tool result or document).

This is conceptually what happens when an agent calls a tool and
incorporates the result into its response.

We simulate it here with a simple weighted lookup.
"""

import numpy as np

def show_cross_attention(query_context, source_text, query_word, scenario):
    """
    Simulate cross-attention: query from generation, keys/values from source.
    Shows which source words the model focuses on when generating query_word.
    """
    source_words = source_text.split()
    np.random.seed(hash(query_word) % 2**31)

    # Simulate relevance scores based on word overlap
    scores = np.array([
        1.8 if w.lower() in query_word.lower() or query_word.lower() in w.lower()
        else 1.2 if any(c in w.lower() for c in query_word.lower())
        else 0.6 + np.random.rand() * 0.4
        for w in source_words
    ])

    # Boost words that share characters with query
    for i, w in enumerate(source_words):
        if w in ['costs', 'cost', 'price', 'prix', '€180', '£', '$'] and 'cost' in query_word.lower():
            scores[i] = 2.5
        if w in ['1889', 'completed', 'built', 'year'] and 'built' in query_word.lower():
            scores[i] = 2.5
        if w in ['330', 'metres', 'tall', 'height'] and 'tall' in query_word.lower():
            scores[i] = 2.5

    e     = np.exp(scores - scores.max())
    probs = e / e.sum()

    col_w   = max(len(w) for w in source_words)
    top_idx = probs.argmax()

    print(f"\n{'═'*60}")
    print(f"  TYPE 5 · CROSS-ATTENTION  |  {scenario}")
    print(f"{'═'*60}")
    print(f"  Source     : \"{source_text}\"")
    print(f"  Query word : \"{query_word}\"  (what the model is currently writing)")
    print(f"  Context    : \"{query_context}\"")
    print()
    print(f"  {'Source word':<{col_w}}   Weight  Attention bar")
    print(f"  {'─'*55}")
    for j, (word, weight) in enumerate(zip(source_words, probs)):
        bar    = '█' * int(weight * 30)
        marker = '  ← focus' if j == top_idx else ''
        print(f"  {word:<{col_w}}   {weight:.3f}   {bar}{marker}")
    print(f"  {'─'*55}")
    print(f"  Model reads \"{source_words[top_idx]}\" most — uses it in the response")
    print()


# Example 1: RAG — model reads retrieved document while answering
show_cross_attention(
    query_context = "The restaurant costs",
    source_text   = "Le Jules Verne Eiffel Tower costs €180 per person open Tuesday",
    query_word    = "costs",
    scenario      = "RAG — model reads retrieved doc while writing answer"
)

# Example 2: Agent tool result
show_cross_attention(
    query_context = "The tower was",
    source_text   = "Eiffel Tower completed 1889 height 330 metres iron lattice structure",
    query_word    = "built",
    scenario      = "Agent tool result — model reads search output"
)

print("💡 Cross-attention is the bridge between the model and external data.")
print("   RAG (Week 4): source = retrieved document chunks")
print("   Agents (Week 6): source = tool call results")
print("   MCP (Week 8): source = standardised tool outputs")
print()
print("   Same mechanism. Different sources. That is the whole architecture.")


════════════════════════════════════════════════════════════
  TYPE 5 · CROSS-ATTENTION  |  RAG — model reads retrieved doc while writing answer
════════════════════════════════════════════════════════════
  Source     : "Le Jules Verne Eiffel Tower costs €180 per person open Tuesday"
  Query word : "costs"  (what the model is currently writing)
  Context    : "The restaurant costs"

  Source word   Weight  Attention bar
  ───────────────────────────────────────────────────────
  Le        0.041   █
  Jules     0.068   ██
  Verne     0.044   █
  Eiffel    0.040   █
  Tower     0.068   ██
  costs     0.248   ███████  ← focus
  €180      0.248   ███████
  per       0.040   █
  person    0.068   ██
  open      0.068   ██
  Tuesday   0.068   ██
  ───────────────────────────────────────────────────────
  Model reads "costs" most — uses it in the response


════════════════════════════════════════════════════════════
  TYPE 5 · CROSS-ATTENTION  |  Agent tool result — model reads search outp

### Attention Types — Summary

In [49]:
"""
One-cell summary of all five attention types and where each appears
in the programme and in real production systems.
"""

print("═" * 70)
print("  ATTENTION TYPES — SUMMARY")
print("═" * 70)
print()

rows = [
    ("1  Self-attention",
     "Every word sees every word",
     "BERT, RAG embeddings",
     "Week 4"),
    ("2  Causal masked",
     "Only sees past tokens",
     "GPT, Claude, Gemini",
     "Every session"),
    ("3  Multi-head",
     "H heads in parallel",
     "All transformers",
     "Every session"),
    ("4  Padding mask",
     "Ignores <PAD> tokens",
     "All batched inference",
     "Always running"),
    ("5  Cross-attention",
     "Query from one, K/V from another",
     "Agents, RAG, MCP",
     "Weeks 4, 6, 7, 8"),
]

print(f"  {'Type':<22} {'What it does':<33} {'Used in':<22} {'Week'}")
print(f"  {'─'*90}")
for type_, what, used, week in rows:
    print(f"  {type_:<22} {what:<33} {used:<22} {week}")

print()
print("  Key insight:")
print("  The context window is the BOX.")
print("  Attention is HOW the model reads what is in the box.")
print("  Everything else in this programme is about WHAT you put in the box.")

══════════════════════════════════════════════════════════════════════
  ATTENTION TYPES — SUMMARY
══════════════════════════════════════════════════════════════════════

  Type                   What it does                      Used in                Week
  ──────────────────────────────────────────────────────────────────────────────────────────
  1  Self-attention      Every word sees every word        BERT, RAG embeddings   Week 4
  2  Causal masked       Only sees past tokens             GPT, Claude, Gemini    Every session
  3  Multi-head          H heads in parallel               All transformers       Every session
  4  Padding mask        Ignores <PAD> tokens              All batched inference  Always running
  5  Cross-attention     Query from one, K/V from another  Agents, RAG, MCP       Weeks 4, 6, 7, 8

  Key insight:
  The context window is the BOX.
  Attention is HOW the model reads what is in the box.
  Everything else in this programme is about WHAT you put in the box

---
## Act 3 · Generation — Seeing Inside Word by Word

Generation is not retrieval. The model constructs every word from scratch by asking:  
*"Given everything I have read so far, what word is most likely to come next?"*

In [50]:
"""
Show generation step by step — print the probability distribution
at each step before picking the next word.
This is exactly what GPT does internally for every token.
"""

def generate_with_inspection(
    prompt_words,
    max_steps    = 8,
    temperature  = 1.0,
    top_k        = 6,
    greedy       = False,
):
    """
    Generate text word by word, showing the top candidates at each step.

    temperature : >1 = more random (creative), <1 = more focused (safe)
    top_k       : only consider this many candidates at each step
    greedy      : always pick the single most likely word
    """
    current = ['<BOS>'] + list(prompt_words)
    generated = list(prompt_words)

    print(f"\n{'═'*58}")
    mode = 'GREEDY' if greedy else f'SAMPLING  temp={temperature}  top-k={top_k}'
    print(f"  GENERATION — {mode}")
    print(f"{'═'*58}")
    print(f"  Prompt: \"{' '.join(prompt_words)}\"")
    print()

    tiny.eval()
    with torch.no_grad():
        for step in range(max_steps):
            ids    = torch.tensor([encode(current)])
            logits = tiny(ids)[0, -1, :]            # last position, all vocab

            # Temperature scaling
            scaled = logits / max(temperature, 1e-5)

            # Top-k filtering
            if top_k:
                vals, _ = torch.topk(scaled, min(top_k, scaled.size(-1)))
                scaled[scaled < vals[-1]] = float('-inf')

            probs = F.softmax(scaled, dim=-1)

            # Top candidates for display
            top_probs, top_ids = torch.topk(probs, min(5, V))
            candidates = [
                (i2v.get(idx.item(), '<UNK>'), p.item())
                for idx, p in zip(top_ids, top_probs)
                if i2v.get(idx.item(), '<UNK>') not in ('<PAD>', '<UNK>')
            ][:4]

            # Pick next word
            if greedy:
                next_word = candidates[0][0] if candidates else '<EOS>'
            else:
                words_c  = [w for w, _ in candidates]
                probs_c  = np.array([p for _, p in candidates])
                probs_c  = probs_c / probs_c.sum()
                next_word = np.random.choice(words_c, p=probs_c)

            if next_word in ('<EOS>', '<PAD>'):
                break

            # Print this step
            context_display = ' '.join(generated[-4:])
            print(f"  Step {step+1}: '...{context_display}' → ?")
            for i, (word, prob) in enumerate(candidates):
                bar    = '█' * int(prob * 20)
                chosen = ' ← CHOSEN' if word == next_word else ''
                print(f"    {word:<14} {prob:5.1%}  {bar}{chosen}")
            print()

            generated.append(next_word)
            current.append(next_word)

    print(f"  Final: \"{' '.join(generated)}\"")
    print(f"{'═'*58}")
    return generated


_ = generate_with_inspection(
    ['once', 'upon', 'a'],
    max_steps   = 6,
    temperature = 0.8,
    top_k       = 5,
)


══════════════════════════════════════════════════════════
  GENERATION — SAMPLING  temp=0.8  top-k=5
══════════════════════════════════════════════════════════
  Prompt: "once upon a"

  Step 1: '...once upon a' → ?
    time           99.6%  ███████████████████ ← CHOSEN
    brilliant       0.2%  
    dark            0.1%  
    <EOS>           0.1%  

  Step 2: '...once upon a time' → ?
    the            99.9%  ███████████████████ ← CHOSEN
    once            0.0%  
    she             0.0%  
    <EOS>           0.0%  

  Step 3: '...upon a time the' → ?
    brave          99.4%  ███████████████████ ← CHOSEN
    wicked          0.3%  
    fantastic       0.2%  
    little          0.1%  

  Step 4: '...a time the brave' → ?
    knight         99.8%  ███████████████████ ← CHOSEN
    princess        0.1%  
    lived           0.0%  
    in              0.0%  

  Step 5: '...time the brave knight' → ?
    rode           99.8%  ███████████████████ ← CHOSEN
    princess        0.1%  
    

In [13]:
"""
Temperature comparison — same prompt, same model, three different dials.
This is the most interactive cell — ask the audience to predict which output
will be most surprising before revealing the high-temperature result.
"""

DEMO_PROMPT = ['the', 'brave', 'knight']

print(f'Prompt: "{" ".join(DEMO_PROMPT)}"')
print()

configs = [
    ("GREEDY          ", dict(greedy=True, max_steps=8)),
    ("temp=0.5  top-k ", dict(temperature=0.5, top_k=5, max_steps=8)),
    ("temp=1.2  top-k ", dict(temperature=1.2, top_k=5, max_steps=8)),
]

print(f"{'Strategy':<20}  Output")
print("─" * 65)

tiny.eval()
for label, kwargs in configs:
    torch.manual_seed(42)
    current = ['<BOS>'] + list(DEMO_PROMPT)
    result  = list(DEMO_PROMPT)

    with torch.no_grad():
        for _ in range(kwargs.get('max_steps', 8)):
            ids    = torch.tensor([encode(current)])
            logits = tiny(ids)[0, -1, :]
            if kwargs.get('greedy'):
                nid = logits.argmax().item()
            else:
                scaled = logits / kwargs.get('temperature', 1.0)
                tk     = kwargs.get('top_k', 0)
                if tk:
                    v, _ = torch.topk(scaled, min(tk, scaled.size(-1)))
                    scaled[scaled < v[-1]] = float('-inf')
                p   = F.softmax(scaled, dim=-1)
                nid = torch.multinomial(p, 1).item()
            nw = i2v.get(nid, '<EOS>')
            if nw in ('<EOS>', '<PAD>', '<BOS>'): break
            result.append(nw)
            current.append(nw)

    print(f"  {label}  {' '.join(result)}")

print()
print("💡 Greedy = always the most probable word. Deterministic. Repetitive.")
print("   Low temp = confident, conservative, predictable.")
print("   High temp = samples from a flatter distribution. Creative but riskier.")
print()
print("   This is the 'temperature' slider in ChatGPT and every LLM API.")
print("   Same model. Same weights. Different dial = different character.")

Prompt: "the brave knight"

Strategy              Output
─────────────────────────────────────────────────────────────────
  GREEDY            the brave knight rode through the dark forest
  temp=0.5  top-k   the brave knight rode through the dark forest
  temp=1.2  top-k   the brave knight rode through the dark gold with her kind

💡 Greedy = always the most probable word. Deterministic. Repetitive.
   Low temp = confident, conservative, predictable.
   High temp = samples from a flatter distribution. Creative but riskier.

   This is the 'temperature' slider in ChatGPT and every LLM API.
   Same model. Same weights. Different dial = different character.


---
## Act 4 · Load Pre-Built Model

The TinyLM trained above learned from 8 sentences.  
My pre-built model trained on 15,000 stories for 2 full epochs.  
Same architecture. Different scale. Same attention mechanism running underneath.

In [51]:
"""
Load the pre-built storyteller model from Hugging Face Hub.
Falls back to local copy if Hub is unreachable — session-proof.

Participants use torch_dtype=torch.float32 explicitly
because the model was trained with float16 on GPU but
Skill Boost runs on CPU which requires float32.
"""
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, time

HF_REPO = "viviktchaudhary/tiny-slm-storyteller-v1"   # my own SLM model

print(f"Loading model: {HF_REPO}")
print("(First load downloads ~1 GB — subsequent loads use cache)")
print()

t0 = time.time()
try:
    slm       = AutoModelForCausalLM.from_pretrained(HF_REPO, dtype=torch.float32)
    slm_tok   = AutoTokenizer.from_pretrained(HF_REPO)
    print(f"✅ Loaded from Hub in {time.time()-t0:.0f}s")
except Exception as e:
    print(f"Hub unavailable ({e})")
    print("Loading local backup...")
    slm       = AutoModelForCausalLM.from_pretrained("./storyteller_merged", dtype=torch.float32)
    slm_tok   = AutoTokenizer.from_pretrained("./storyteller_merged")
    print(f"✅ Loaded from local backup in {time.time()-t0:.0f}s")

slm.eval()
params = sum(p.numel() for p in slm.parameters())
print()
print(f"Model parameters : {params:,}")
print(f"Trained on       : 15,000 TinyStories + Gutenberg fairy tales")
print(f"Training method  : LoRA fine-tune of Qwen2.5-0.5B")
print()
print("Compare:")
print(f"  Our TinyLM (8 sentences)  :     {sum(p.numel() for p in tiny.parameters()):>12,} parameters")
print(f"  This model (15k stories)  :     {params:>12,} parameters")
print(f"  GPT-2 small               :    117,000,000 parameters")
print(f"  GPT-4 (estimated)         : ~1,000,000,000,000 parameters")

Loading model: viviktchaudhary/tiny-slm-storyteller-v1
(First load downloads ~1 GB — subsequent loads use cache)



Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✅ Loaded from Hub in 7s

Model parameters : 494,032,768
Trained on       : 15,000 TinyStories + Gutenberg fairy tales
Training method  : LoRA fine-tune of Qwen2.5-0.5B

Compare:
  Our TinyLM (8 sentences)  :           75,520 parameters
  This model (15k stories)  :      494,032,768 parameters
  GPT-2 small               :    117,000,000 parameters
  GPT-4 (estimated)         : ~1,000,000,000,000 parameters


In [52]:
"""
Generation function for the pre-built model.
Uses the same three strategies shown with TinyLM —
the audience can compare outputs directly.
"""

def tell_story(
    prompt,
    max_new_tokens     = 150,
    temperature        = 0.85,
    top_p              = 0.92,
    top_k              = 50,
    repetition_penalty = 1.15,
    greedy             = False,
):
    """
    Generate a story from a prompt using the pre-built SLM.

    temperature        : 0.7 = safe, 1.2 = creative
    repetition_penalty : >1.0 prevents the model repeating phrases
    greedy             : if True, always picks most likely word
    """
    full_prompt = f"<story>\n{prompt}"
    inputs      = slm_tok(full_prompt, return_tensors="pt")

    with torch.no_grad():
        output_ids = slm.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = temperature if not greedy else 1.0,
            top_p              = top_p,
            top_k              = top_k,
            repetition_penalty = repetition_penalty,
            do_sample          = not greedy,
            pad_token_id       = slm_tok.eos_token_id,
        )

    full_text = slm_tok.decode(output_ids[0], skip_special_tokens=True)

    # Clean story tags and return only the generated part
    story = full_text.replace('<story>', '').replace('</story>', '')
    for marker in ['</s>', '<|endoftext|>', '<story>']:
        if marker in story:
            story = story[:story.index(marker)]

    return story.strip()


# Test with the same prompts used throughout the session
DEMO_PROMPTS = [
    "Once upon a time there was a little girl who lived near a dark forest.",
    "The brave knight rode through the enchanted kingdom and discovered",
    "Deep in the ocean where the water was as blue as the sky,",
]

print("=" * 65)
print("PRE-BUILT MODEL — Story Generation")
print("=" * 65)

for i, prompt in enumerate(DEMO_PROMPTS, 1):
    story = tell_story(prompt, max_new_tokens=120, temperature=0.85)
    print(f"\n[{i}] '{prompt[:50]}...'")
    print(f"    {story[:300]}")
    if len(story) > 300:
        print("    [...]")
    print()

PRE-BUILT MODEL — Story Generation

[1] 'Once upon a time there was a little girl who lived...'
    Once upon a time there was a little girl who lived near a dark forest. Every day she would run and play in the forest with her friends.

One morning, the girl found something strange in the forest: an old man's car! The man had been driving his car for many years without ever leaving it behind.
 
Th
    [...]


[2] 'The brave knight rode through the enchanted kingdo...'
    The brave knight rode through the enchanted kingdom and discovered a special treasure chest. Inside were magical objects that made his enemies disappear!

He then had to explore the forest, but one of the tree branches was too high for him. So he climbed up a ladder, eventually reaching the top. 

I
    [...]


[3] 'Deep in the ocean where the water was as blue as t...'
    Deep in the ocean where the water was as blue as the sky, there lived a little fish named Fred. He had three friends: a big dolphin and two colorf

---
## Act 5 · Before vs After — Scale Makes the Difference

In [53]:
"""
Side-by-side comparison: TinyLM (8 training sentences) vs
pre-built model (25,000 stories, 2 epochs).
Same architecture. Same attention mechanism. Different scale.
"""

COMPARE_PROMPT = "Once upon a time there was a little girl who lived near a dark forest."

print(f'Prompt: "{COMPARE_PROMPT}"')
print()

# TinyLM output
tiny.eval()
current = ['<BOS>'] + COMPARE_PROMPT.lower().split()
tiny_result = COMPARE_PROMPT.lower().split()
with torch.no_grad():
    for _ in range(12):
        ids    = torch.tensor([encode(current)])
        logits = tiny(ids)[0, -1, :]
        logits = logits / 0.8
        kv, _ = torch.topk(logits, 5)
        logits[logits < kv[-1]] = float('-inf')
        nid = torch.multinomial(F.softmax(logits, dim=-1), 1).item()
        nw  = i2v.get(nid, '<EOS>')
        if nw in ('<EOS>', '<PAD>', '<BOS>'): break
        tiny_result.append(nw)
        current.append(nw)

# Pre-built model output
slm_result = tell_story(COMPARE_PROMPT, max_new_tokens=80, temperature=0.85)

print("─" * 65)
print("TinyLM  (trained on 8 sentences for 30 epochs):")
print(f"  {' '.join(tiny_result)}")
print()
print("Pre-built SLM  (trained on 25,000 stories for 2 epochs):")
print(f"  {slm_result[:300]}")
print("─" * 65)
print()
print("Same attention mechanism. Same causal mask. Same generation logic.")
print("The only difference: scale of training data and model parameters.")
print()
print("This gap — TinyLM to this model — is a factor of ~7,000× in parameters.")
print("GPT-3 to this model is a factor of ~350× on top of that.")
print("The architecture you now understand runs at every scale.")

Prompt: "Once upon a time there was a little girl who lived near a dark forest."

─────────────────────────────────────────────────────────────────
TinyLM  (trained on 8 sentences for 30 epochs):
  once upon a time there was a little girl who lived near a dark forest. the water

Pre-built SLM  (trained on 25,000 stories for 2 epochs):
  Once upon a time there was a little girl who lived near a dark forest. She liked to explore it, but one day she stumbled into an open hole in the ground. It looked so mysterious!

The little girl thought for a moment before taking some brave steps forward.

"What shall I do?" she asked herself.

Sud
─────────────────────────────────────────────────────────────────

Same attention mechanism. Same causal mask. Same generation logic.
The only difference: scale of training data and model parameters.

This gap — TinyLM to this model — is a factor of ~7,000× in parameters.
GPT-3 to this model is a factor of ~350× on top of that.
The architecture you now under

---
## Act 6 · Prompt Playground

**Your turn.** Change the prompt and temperature below and run the cell.

In [17]:
# ✏️  Change these and re-run — try different prompts and temperatures!

MY_PROMPT      = "The old wizard looked at the young girl and said"
MY_TEMPERATURE = 1.0      # 0.7 = safe  |  1.0 = balanced  |  1.3 = creative
MY_LENGTH      = 120      # number of words to generate

print(f'Prompt      : "{MY_PROMPT}"')
print(f'Temperature : {MY_TEMPERATURE}')
print(f'Length      : {MY_LENGTH} tokens')
print()

story = tell_story(MY_PROMPT, max_new_tokens=MY_LENGTH, temperature=MY_TEMPERATURE)
print(story)
print()
print("─" * 65)
print("Run again for a different story (temperature > 0.7 adds randomness).")
print("Change MY_TEMPERATURE to 0.3 for a more predictable output.")
print("Change MY_TEMPERATURE to 1.5 for a more unpredictable output.")

Prompt      : "The old wizard looked at the young girl and said"
Temperature : 1.0
Length      : 120 tokens

The old wizard looked at the young girl and said, â€œYou are my disciple. Do you know what is meant by â€˜pioneer?â€™ What do we mean when we say that a person has made great progress in life?"
The little girl smiled as she thought about it. She had been working hard since she was very small.
"Indeed," she replied.

"What does it take to have such a big effect on others?" asked the old wizard.

"The idea behind this is just being kind and helping people who need it most."

The old wizard nodded his head with approval. He then said, "I'm sorry to

─────────────────────────────────────────────────────────────────
Run again for a different story (temperature > 0.7 adds randomness).
Change MY_TEMPERATURE to 0.3 for a more predictable output.
Change MY_TEMPERATURE to 1.5 for a more unpredictable output.


In [ ]:
# Audience prompt cell — type any opening line
# Suggested prompts to try:
#   "Once upon a time a dragon fell in love with"
#   "The princess climbed to the top of the tower and"
#   "Deep in the enchanted forest where the trees whispered"
#   "The terrible witch cackled and said"

AUDIENCE_PROMPT = "Once upon a time a dragon fell in love with"  

for temp, label in [(0.7, "Safe"), (1.0, "Balanced"), (1.3, "Creative")]:
    story = tell_story(AUDIENCE_PROMPT, max_new_tokens=60, temperature=temp)
    print(f"[{label:<10} temp={temp}]")
    print(f"  {story[:200]}")
    print()

---
## Act 7 · Bridge to What Comes Next

You now have the mental model. Every remaining week is an application of the same ideas.

In [ ]:
"""
Preview of Weeks 4–8 — each shown as a single concept cell.
Not deep dives. Just enough to make the next session feel inevitable.
"""

print("═" * 65)
print("  WEEK 4 PREVIEW — RAG: Fill the Box with Real Data")
print("═" * 65)
print()
print("The problem we opened with: the AI hallucinated a restaurant.")
print()
print("The fix:")

# Simulate RAG
knowledge_base = [
    "Le Jules Verne: located on 2nd floor of Eiffel Tower. Open Tue-Sun. Prix fixe €250.",
    "Café de l'Esplanade: 52 Rue Fabert, 75007. Open daily. Average €45 per person.",
    "Brasserie Thoumieux: 79 Rue Saint-Dominique. Michelin starred. Open for dinner.",
]

user_question = "Recommend a restaurant near the Eiffel Tower"

print()
print("  [RETRIEVAL] User asked:", user_question)
print("  [RETRIEVAL] Found 3 relevant documents:")
for i, doc in enumerate(knowledge_base, 1):
    print(f"    {i}. {doc}")
print()
print("  [CONTEXT WINDOW — now filled with real data]:")
print("    SYSTEM: Answer using only the provided documents.")
for doc in knowledge_base:
    print(f"    RETRIEVED: {doc}")
print(f"    USER: {user_question}")
print()
print("  [MODEL RESPONSE — grounded, not hallucinated]:")
print("    'Based on your location near the Eiffel Tower, Le Jules Verne")
print("     is directly on the tower's 2nd floor (open Tue–Sun, prix fixe €250).")
print("     For a more casual option, Brasserie Thoumieux is nearby and Michelin starred.'")
print()
print("  ✅ Same LLM. Same attention. Different context window.")
print("     Week 4 builds the retrieval system that fills this window.")

In [ ]:
"""
Agent preview — tool call loop.
The model can now decide to call tools and put results in the context window.
"""

print("═" * 65)
print("  WEEK 6 PREVIEW — Agents: The Model Fills Its Own Box")
print("═" * 65)
print()
print("In RAG: YOU retrieve and fill the context window.")
print("In Agents: THE MODEL decides what to retrieve and fills it itself.")
print()

# Simulate agent loop
def fake_search(query):
    """Pretend this is a real search tool."""
    return f"[Search result for '{query}']: Le Jules Verne, 2nd floor Eiffel Tower, costs €250 prix fixe, open Tue-Sun."

print("  User: 'Find me a restaurant near the Eiffel Tower and book a table for 2 on Friday'")
print()
print("  [AGENT TURN 1]: Model decides to search")
print("    → Calls: search('restaurants near Eiffel Tower Paris')")
tool_result = fake_search("restaurants near Eiffel Tower Paris")
print(f"    ← Tool returns: {tool_result}")
print()
print("  [AGENT TURN 2]: Model puts tool result in context window")
print("    Context now contains: user question + search result")
print("    → Calls: book_table('Le Jules Verne', date='Friday', party=2)")
print("    ← Tool returns: 'Reservation confirmed for Friday 8pm, table for 2'")
print()
print("  [FINAL RESPONSE]:")
print("    'I have booked Le Jules Verne on the Eiffel Tower's 2nd floor")
print("     for Friday at 8pm, table for 2. Confirmation reference: #LJV2847.'")
print()
print("  ✅ Cross-attention you saw earlier is how the model reads each tool result.")
print("     Week 6 builds this full agent loop.")
print()
print("═" * 65)
print("  WEEK 8 PREVIEW — MCP: Standardised Tool Connections")
print("═" * 65)
print()
print("  The agent loop needs tools. MCP is the standard plug.")
print("  Any tool that speaks MCP can connect to any agent that speaks MCP.")
print("  Think USB — one standard connector, any device.")
print("  Week 8 is about building those plugs and connecting them.")

---
## Session Close

**Return to the opening question:**  
*"I asked the AI to recommend a restaurant near the Eiffel Tower and it made one up. Why?"*

You can now answer it completely:

1. The model's **context window** contained only your question — no restaurant data
2. **Attention** read your question and found no grounding facts to connect to
3. **Generation** filled the gap with the most probable-sounding words from training
4. The result was plausible — but not real. That is hallucination by design, not by accident.

**The fix — which you now know is "RAG" in the Week 4.**
---

## What was covered Today

```
Context window  ── the box everything goes through
     │
     ├── Self-attention      ── how words connect to each other
     ├── Causal mask         ── why generation is left-to-right
     ├── Multi-head          ── parallel reading of different patterns
     ├── Padding mask        ── ignoring empty slots
     └── Cross-attention     ── reading external sources (RAG, agents, MCP)
     │
Generation  ─── word-by-word probability sampling
     │
     ├── Greedy             ── most likely word, always
     ├── Temperature        ── flatten or sharpen the distribution
     └── Top-k / Nucleus    ── constrain the candidate set
```

**Next week: RAG — we fill the box with your own data.**